# Extensions — Google Colab

**Project:** Vision-Language Models in Radiology

This notebook covers all three research extensions:

| Extension | Description | Prerequisite |
|-----------|-------------|-------------|
| **Ext 1** | Uncertainty-Aware Classification (MC Dropout + Conformal Prediction) | CheXzero best.pt + NIH-14 |
| **Ext 2** | Factuality-Constrained Generation (Proxy RadGraph loss fine-tuning) | R2Gen best.pt + IU X-Ray |
| **Ext 3** | Cross-Dataset Robustness (zero-shot → few-shot on NIH-14) | CheXzero best.pt + NIH-14 |

> **Runtime:** T4 GPU.  
> **Prerequisites:** Run `chexzero_colab.ipynb` and/or `r2gen_colab.ipynb` first to produce the required checkpoints.

---
## 0. Setup

In [ ]:
import os, sys, logging

REPO_URL = "https://github.com/SinaDns/radiology-vision-language-models.git"
REPO_DIR = "/content/radiology-vision-language-models"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

!pip install -q -r requirements.txt

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    datefmt='%H:%M:%S',
)

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── Mount Google Drive to load checkpoints ────────────────────────────────────
# Uncomment if checkpoints are saved to Drive:
# from google.colab import drive
# drive.mount("/content/drive")

# Set checkpoint paths
CHEXZERO_CKPT = "experiments/results/checkpoints/chexzero/best.pt"
R2GEN_CKPT    = "experiments/results/checkpoints/r2gen/best.pt"
VOCAB_PATH    = "experiments/results/r2gen_vocab.json"

# Override if loading from Drive:
# CHEXZERO_CKPT = "/content/drive/MyDrive/chexzero_best.pt"
# R2GEN_CKPT    = "/content/drive/MyDrive/r2gen_best.pt"

import os
for p in [CHEXZERO_CKPT, R2GEN_CKPT]:
    if os.path.exists(p):
        print(f"✓ Found: {p}")
    else:
        print(f"✗ Missing: {p}  (run the prerequisite notebook first)")

---
## Extension 1: Uncertainty-Aware Classification

**Goal:** Quantify epistemic uncertainty with MC Dropout; calibrate predictions with conformal prediction to guarantee 90% coverage.

**Targets:** ECE reduction 20-30%, coverage ≥ 90% at α=0.10.

**Prerequisites:** CheXzero best.pt, NIH-14 dataset (see Section 5 of `chexzero_colab.ipynb`).

In [ ]:
from src.models.chexzero import CheXzero
from src.models.uncertainty import MCDropoutCheXzero, ConformalPredictor

base_model = CheXzero(embed_dim=512)
ckpt = torch.load(CHEXZERO_CKPT, map_location=device)
base_model.load_state_dict(ckpt["model_state_dict"])

mc_model = MCDropoutCheXzero(
    chexzero=base_model,
    dropout_rate=0.1,
    n_samples=10,   # 10 MC passes is sufficient; 20 may OOM on T4
)
mc_model.to(device)
print(f"MC Dropout model ready — epoch {ckpt.get('epoch', '?')}")

In [ ]:
# Pre-compute text embeddings
from transformers import AutoTokenizer
from src.evaluation.zero_shot import PATHOLOGY_PROMPTS, _encode_prompts
from src.data_loaders.nih_chestxray14 import NIH14_LABELS

tokenizer_bert = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

text_embs = []
for label in NIH14_LABELS:
    pos_list = PATHOLOGY_PROMPTS.get(label, {}).get(
        "positive", [f"findings consistent with {label.lower()}"]
    )
    embs = _encode_prompts(base_model, pos_list, tokenizer_bert, device)
    text_embs.append(embs.mean(0, keepdim=True))
text_embs_t = torch.cat(text_embs, 0).to(device)
print(f"Text embeddings: {text_embs_t.shape}")

In [ ]:
# NIH-14 test set — update path as needed
NIH14_DIR  = "/content/radiology-vision-language-models/data/nih_chestxray14"
TEST_CSV   = f"{NIH14_DIR}/test_list.txt"
CALIB_CSV  = f"{NIH14_DIR}/train_val_list.txt"

import os
if not os.path.exists(NIH14_DIR + "/images"):
    print("NIH-14 not found. Follow Section 5 of chexzero_colab.ipynb to download.")
else:
    from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset
    from src.data_loaders.transforms import get_val_transforms
    from torch.utils.data import DataLoader

    test_ds  = NIHChestXray14Dataset(NIH14_DIR, split="test",  split_csv=TEST_CSV,  transform=get_val_transforms(320))
    calib_ds = NIHChestXray14Dataset(NIH14_DIR, split="train", split_csv=CALIB_CSV, transform=get_val_transforms(320))
    # batch_size=32 for T4 safety; increase to 64 for A100
    test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    calib_loader = DataLoader(calib_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    print(f"Test: {len(test_ds)}  Calibration: {len(calib_ds)}")

In [ ]:
import numpy as np

# MC Dropout scoring on test set
all_mean, all_unc, all_labels = [], [], []

for batch_idx, batch in enumerate(test_loader):
    images = batch["image"].to(device)
    out = mc_model.predict_with_uncertainty(images, text_embs_t, n_samples=10)
    all_mean.append(out["mean_scores"].cpu().numpy())
    all_unc.append(out["uncertainty"].cpu().numpy())
    all_labels.append(batch["labels"].numpy())
    if (batch_idx+1) % 50 == 0:
        print(f"Scored {batch_idx+1}/{len(test_loader)} batches")

mean_scores = np.concatenate(all_mean)
uncertainty = np.concatenate(all_unc)
labels_np   = np.concatenate(all_labels)
print(f"Scored {len(labels_np)} test images")
print(f"Mean uncertainty: {uncertainty.mean():.4f}")

In [ ]:
from src.evaluation.metrics import compute_auroc, classification_report

scores_dict = {l: mean_scores[:, i] for i, l in enumerate(NIH14_LABELS)}
labels_dict = {l: labels_np[:, i]   for i, l in enumerate(NIH14_LABELS)}
auroc = compute_auroc(scores_dict, labels_dict)

print("=== MC Dropout Zero-Shot AUROC ===")
print(classification_report(mean_scores, labels_np, label_names=NIH14_LABELS))
print(f"\nMean AUROC: {auroc['mean_auroc']:.4f}")

In [ ]:
# Conformal prediction calibration
calib_scores_list, calib_labels_list = [], []
for batch in calib_loader:
    imgs = batch["image"].to(device)
    with torch.no_grad():
        emb = base_model.encode_image(imgs).cpu()
        sc  = (emb @ text_embs_t.cpu().T).numpy()
    calib_scores_list.append(sc)
    calib_labels_list.append(batch["labels"].numpy())

calib_scores = np.concatenate(calib_scores_list)
calib_labels = np.concatenate(calib_labels_list)

predictor = ConformalPredictor(alpha=0.1)
predictor.calibrate(calib_scores, calib_labels)

coverage = predictor.compute_coverage(mean_scores, labels_np)
ece_stats = predictor.compute_ece_reduction(calib_scores, mean_scores, labels_np)

print(f"\nConformal Coverage: {coverage['coverage']:.4f}  (target ≥ 0.90)")
print(f"Avg prediction set size: {coverage['avg_set_size']:.2f} / 14")
print(f"ECE before: {ece_stats['ece_before']:.4f}")
print(f"ECE after:  {ece_stats['ece_after']:.4f}")
print(f"ECE reduction: {ece_stats['ece_reduction']:.1%}  (target 20-30%)")

In [ ]:
# Uncertainty visualisation
import matplotlib.pyplot as plt
import os

os.makedirs("experiments/results", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: per-class uncertainty
mean_unc_per_class = uncertainty.mean(axis=0)
axes[0].barh(NIH14_LABELS, mean_unc_per_class, color='salmon')
axes[0].set_xlabel('Mean Epistemic Uncertainty')
axes[0].set_title('Per-Class Uncertainty (MC Dropout)')

# Right: AUROC vs uncertainty scatter
auroc_vals = [auroc.get(l, 0) for l in NIH14_LABELS]
axes[1].scatter(mean_unc_per_class, auroc_vals, color='steelblue', s=60)
for i, l in enumerate(NIH14_LABELS):
    axes[1].annotate(l, (mean_unc_per_class[i], auroc_vals[i]), fontsize=7)
axes[1].set_xlabel('Mean Uncertainty')
axes[1].set_ylabel('AUROC')
axes[1].set_title('AUROC vs Uncertainty')

plt.tight_layout()
plt.savefig('experiments/results/ext1_uncertainty.png', dpi=150)
plt.show()
print("Plot saved to experiments/results/ext1_uncertainty.png")

---
## Extension 2: Factuality-Constrained Generation

**Goal:** Fine-tune R2Gen with a proxy clinical factuality penalty to reduce hallucinations.

**Target:** Proxy RadGraph F1 improvement after factuality fine-tuning.

**Prerequisite:** R2Gen best.pt, IU X-Ray dataset.

In [ ]:
import os
from src.data_loaders.iu_xray_seq2seq import build_tokenizer

tok = build_tokenizer(
    data_dir="data/iu_xray/",
    save_path=VOCAB_PATH,
    min_freq=3,
)
print(f"Vocabulary: {tok.vocab_size} tokens")

In [ ]:
from src.utils.config import load_config

fact_config = load_config("experiments/configs/factuality.yaml")
fact_config["model"]["r2gen_checkpoint"] = R2GEN_CKPT
fact_config["paths"]["iu_xray_dir"]    = "data/iu_xray/"
fact_config["paths"]["checkpoint_dir"] = "experiments/results/checkpoints/factuality/"
fact_config["paths"]["log_dir"]        = "experiments/results/logs/"
fact_config["tokenizer"]["vocab_save_path"] = VOCAB_PATH
fact_config["training"]["batch_size"]   = 8
fact_config["training"]["epochs"]       = 5   # short fine-tune
fact_config["data"]["num_workers"]      = 2
print("Factuality config ready.")

In [ ]:
import torch, os
import torch.amp
from torch.utils.data import DataLoader

from src.data_loaders.iu_xray_seq2seq import IUXraySeq2SeqDataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
from src.models.r2gen import R2GenModel
from src.training.factuality_loss import ProxyFactualityLoss, compute_radgraph_f1
from src.evaluation.generation_metrics import compute_all_metrics, generation_report
from src.utils.logging_utils import setup_logger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_use_amp = torch.cuda.is_available()
logger_fact = setup_logger(fact_config["paths"]["log_dir"], "factuality")

train_ds_f = IUXraySeq2SeqDataset(
    data_dir="data/iu_xray/", tokenizer=tok, split="train",
    val_fraction=0.1, transform=get_train_transforms(224), max_length=100,
)
val_ds_f = IUXraySeq2SeqDataset(
    data_dir="data/iu_xray/", tokenizer=tok, split="val",
    val_fraction=0.1, transform=get_val_transforms(224), max_length=100,
)

train_loader_f = DataLoader(train_ds_f, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader_f   = DataLoader(val_ds_f,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

mcfg = fact_config["model"]
fact_model = R2GenModel(
    vocab_size=tok.vocab_size,
    d_model=mcfg["d_model"], num_heads=mcfg["num_heads"],
    num_enc_layers=mcfg["num_enc_layers"], num_dec_layers=mcfg["num_dec_layers"],
    dim_ff=mcfg["dim_ff"], dropout=mcfg["dropout"],
    num_mem_slots=mcfg["num_mem_slots"], max_seq_len=mcfg["max_seq_len"],
    pretrained_image=False, pad_id=tok.pad_id,
)

if os.path.exists(R2GEN_CKPT):
    r2_ckpt = torch.load(R2GEN_CKPT, map_location=device)
    fact_model.load_state_dict(r2_ckpt["model_state_dict"])
    print(f"Loaded R2Gen weights (epoch {r2_ckpt.get('epoch','?')})")
else:
    print("WARNING: R2Gen checkpoint not found — training from scratch")

fact_model.to(device)
criterion = ProxyFactualityLoss(vocab=tok.word2idx, coverage_weight=0.3, pad_id=tok.pad_id)
optimizer = torch.optim.AdamW(fact_model.parameters(), lr=1e-5, weight_decay=1e-4)
scaler = torch.amp.GradScaler(device=device.type, enabled=_use_amp)

os.makedirs("experiments/results/checkpoints/factuality", exist_ok=True)
os.makedirs("experiments/results", exist_ok=True)
print("Factuality fine-tuning ready.")

In [ ]:
# ── Fine-tuning loop ──────────────────────────────────────────────────────────
import time

best_val = float("inf")
epochs = fact_config["training"]["epochs"]

for epoch in range(epochs):
    fact_model.train()
    total, n = 0.0, 0
    t0 = time.time()

    for i, batch in enumerate(train_loader_f):
        imgs  = batch["image"].to(device)
        inp   = batch["input_ids"].to(device)
        tgt   = batch["target_ids"].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device.type, enabled=_use_amp):
            logits = fact_model(imgs, inp)
            loss, ce, fact_l = criterion(logits, tgt)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(fact_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item(); n += 1
        if (i+1) % 50 == 0:
            logger_fact.info(f"Epoch {epoch} [{i+1}/{len(train_loader_f)}] total={loss:.4f} ce={ce:.4f} fact={fact_l:.4f}")

    # Val
    fact_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader_f:
            imgs = batch["image"].to(device)
            inp  = batch["input_ids"].to(device)
            tgt  = batch["target_ids"].to(device)
            with torch.amp.autocast(device_type=device.type, enabled=_use_amp):
                logits = fact_model(imgs, inp)
                loss, _, _ = criterion(logits, tgt)
            val_loss += loss.item()
    val_loss /= len(val_loader_f)

    logger_fact.info(f"Epoch {epoch} done — train={total/n:.4f} val={val_loss:.4f} time={time.time()-t0:.1f}s")
    print(f"Epoch {epoch}/{epochs-1} — train_loss={total/n:.4f} val_loss={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save({"epoch": epoch, "model_state_dict": fact_model.state_dict()},
                   "experiments/results/checkpoints/factuality/best.pt")
        print(f"  Saved new best checkpoint")

print(f"\nFactuality fine-tuning done — best val loss: {best_val:.4f}")

In [ ]:
# Compare base R2Gen vs factuality-finetuned
import torch
from src.models.r2gen import R2GenModel

def generate_reports(model, loader, tokenizer, device, max_reports=200):
    model.eval()
    hyps, refs = [], []
    for batch in loader:
        if len(hyps) >= max_reports:
            break
        imgs = batch["image"].to(device)
        seqs = model.generate(imgs, bos_id=tokenizer.bos_id,
                               eos_id=tokenizer.eos_id, beam_size=3, max_length=100)
        for s in seqs:
            hyps.append(tokenizer.decode(s))
        refs.extend(batch["report"])
    return hyps[:max_reports], refs[:max_reports]

print("Generating from factuality model…")
hyps_fact, refs_fact = generate_reports(fact_model, val_loader_f, tok, device)
metrics_fact = compute_all_metrics(hyps_fact, refs_fact)
rg_fact = compute_radgraph_f1(hyps_fact, refs_fact)

print("\n=== Factuality Fine-tuned R2Gen ===")
print(generation_report(metrics_fact))
for k, v in rg_fact.items():
    print(f"{k}: {v:.4f}")

---
## Extension 3: Cross-Dataset Robustness

**Goal:** Evaluate zero-shot transfer from IU X-Ray to NIH-14, then recover performance with few-shot linear probes.

**Target:** Few-shot probe recovers 50-70% of the cross-dataset performance drop.

**Prerequisite:** CheXzero best.pt, NIH-14 dataset.

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader

from src.models.chexzero import CheXzero
from src.data_loaders.iu_xray import IUXrayDataset
from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset, NIH14_LABELS
from src.data_loaders.transforms import get_val_transforms
from src.evaluation.cross_dataset import (
    extract_embeddings, compute_domain_gap,
    few_shot_adapt, evaluate_probe, cross_dataset_report,
)
from src.evaluation.zero_shot import compute_zero_shot_scores
from src.evaluation.metrics import compute_auroc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load CheXzero
czero = CheXzero(embed_dim=512)
ckpt  = torch.load(CHEXZERO_CKPT, map_location=device)
czero.load_state_dict(ckpt["model_state_dict"])
czero.to(device).eval()
print(f"CheXzero loaded (epoch {ckpt.get('epoch','?')})")

# IU X-Ray val embeddings (source domain)
iu_ds = IUXrayDataset("data/iu_xray/", split="val", transform=get_val_transforms(320))
iu_loader = DataLoader(iu_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Extracting IU X-Ray embeddings…")
iu_embs = extract_embeddings(czero, iu_loader, device)
print(f"IU X-Ray embeddings: {iu_embs.shape}")

In [ ]:
NIH14_DIR = "/content/radiology-vision-language-models/data/nih_chestxray14"
TEST_CSV  = f"{NIH14_DIR}/test_list.txt"

if not os.path.exists(NIH14_DIR + "/images"):
    print("NIH-14 not found. Follow Section 5 of chexzero_colab.ipynb to download.")
else:
    nih_test_ds  = NIHChestXray14Dataset(NIH14_DIR, split="test", split_csv=TEST_CSV,
                                          transform=get_val_transforms(320))
    nih_loader   = DataLoader(nih_test_ds, batch_size=32, shuffle=False,  # T4-safe
                              num_workers=2, pin_memory=True)

    print("Extracting NIH-14 embeddings…")
    nih_embs = extract_embeddings(czero, nih_loader, device)

    # Collect labels
    all_labels = []
    for batch in nih_loader:
        all_labels.append(batch["labels"].numpy())
    nih_labels_np = np.concatenate(all_labels)
    print(f"NIH-14 embeddings: {nih_embs.shape}  labels: {nih_labels_np.shape}")

In [ ]:
# Domain gap analysis
print("=== Domain Gap (IU X-Ray → NIH-14) ===")
gap = compute_domain_gap(iu_embs, nih_embs)
for k, v in gap.items():
    print(f"  {k:<25}: {v:.4f}")

In [ ]:
# Zero-shot AUROC
print("Computing zero-shot scores…")
zs_scores = compute_zero_shot_scores(czero, nih_loader, NIH14_LABELS, device)
labels_dict = {l: nih_labels_np[:, i] for i, l in enumerate(NIH14_LABELS)}
zs_auroc = compute_auroc(zs_scores, labels_dict)
print(f"Zero-shot mean AUROC: {zs_auroc['mean_auroc']:.4f}")

In [ ]:
# Few-shot linear probe adaptation
nih_labels_t = torch.tensor(nih_labels_np)
K_SHOTS = [1, 5, 10, 25]
results_by_k = {}

for k in K_SHOTS:
    print(f"\nFew-shot k={k}…")
    probe = few_shot_adapt(
        embeddings=nih_embs, labels=nih_labels_t,
        k_shot=k, n_classes=14, n_epochs=100, lr=1e-3, device=device,
    )
    probe_auroc = evaluate_probe(probe, nih_embs, nih_labels_np, device, NIH14_LABELS)
    results_by_k[k] = probe_auroc
    print(f"  k={k} mean AUROC: {probe_auroc['mean_auroc']:.4f}")

In [ ]:
# Print comparison tables
from src.evaluation.cross_dataset import cross_dataset_report

for k in K_SHOTS:
    print(f"\n{'='*60}")
    print(f" k={k} few-shot probe")
    print(cross_dataset_report(zs_auroc, results_by_k[k], NIH14_LABELS))

In [ ]:
# Visualise few-shot recovery curve
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("experiments/results", exist_ok=True)

zs_mean = zs_auroc["mean_auroc"]
fs_means = [results_by_k[k].get("mean_auroc", float("nan")) for k in K_SHOTS]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_SHOTS, fs_means, marker="o", color="steelblue", label="Few-shot probe")
ax.axhline(zs_mean, color="red", linestyle="--", label=f"Zero-shot baseline ({zs_mean:.3f})")
ax.set_xlabel("k (labelled examples per class)")
ax.set_ylabel("Mean AUROC")
ax.set_title("Cross-Dataset Adaptation: IU X-Ray -> NIH-14")
ax.legend()
ax.set_xscale("log")
plt.tight_layout()
plt.savefig("experiments/results/ext3_cross_dataset.png", dpi=150)
plt.show()
print("Plot saved to experiments/results/ext3_cross_dataset.png")